# 3) Calling Functions — Exercises

**Goals:** positional vs keyword calls, unpacking (`*`, `**`), higher-order functions (functions as values), callbacks, `key=` usage, lambdas vs named functions, error wrappers.

### Warm-ups

1. **Call with keywords**

```python
def rect(w, h): return w*h
def rect_kw(**kwargs):
    """Call rect using **kwargs expected to have w and h."""
    ...
assert rect_kw(w=3, h=4) == 12
```

2. \**Call with *args**

```python
def hypotenuse(a, b):
    return (a*a + b*b) ** 0.5
def hypot_from_pair(pair):
    """Call hypotenuse using *pair."""
    ...
assert abs(hypot_from_pair((3,4)) - 5) < 1e-9
```

3. **Sort with key**

```python
def sort_by_last_char(words):
    """Return new list sorted by last character; stable; do not mutate input."""
    ...
assert sort_by_last_char(["ab","aa","bb"]) == ["aa","ab","bb"]
```

### Core

4. **Apply pipeline**

```python
def apply_pipeline(x, funcs):
    """
    funcs: iterable of callables f: x -> x
    Return result of applying them in order.
    """
    ...
assert apply_pipeline(2, [lambda x: x+3, lambda x: x*10]) == 50
```

5. **Filter + map with callables**

```python
def transform(nums, pred, mapper):
    """Keep items where pred(x) is True, then map with mapper(x)."""
    ...
assert transform([1,2,3,4], lambda x:x%2==0, lambda x:x*x) == [4,16]
```

6. **Group with key function**

```python
def group_by(xs, key_fn):
    """Return dict key->list of items in original order."""
    ...
g = group_by(["aa","b","ccc","dd"], key_fn=len)
assert g == {2:["aa","dd"],1:["b"],3:["ccc"]}
```

7. **Retry wrapper**

```python
def retry(fn, *, attempts=3):
    """
    Call fn(); if it raises, retry up to attempts times.
    Return fn() result or raise last error.
    """
    ...
count = {"n":0}
def flaky():
    count["n"] += 1
    if count["n"] < 2: raise RuntimeError("boom")
    return "ok"
assert retry(flaky, attempts=3) == "ok"
```

8. **Decorator: timeit**

```python
import time
def timeit(fn):
    """Decorator returning (result, elapsed_seconds)."""
    ...
@timeit
def slow_add():
    t0 = time.time()
    while time.time() - t0 < 0.01: pass
    return 42
res, secs = slow_add()
assert res == 42 and secs >= 0
```

9. **Caching (simple memoize)**

```python
def memoize(fn):
    """Decorator caching fn(x) for hashable x."""
    ...
calls = {"n":0}
@memoize
def square(x):
    calls["n"] += 1
    return x*x
assert square(10)==100 and square(10)==100 and calls["n"]==1
```

### Challenge

10. **Compose callables**

```python
def compose(*funcs):
    """
    Return f(x) that applies funcs right-to-left: compose(f,g,h)(x) == f(g(h(x))).
    If no funcs, return identity.
    """
    ...
id_fn = compose()
assert id_fn(5) == 5
f = compose(lambda x:x+1, lambda x:x*2, lambda x:x-3)
assert f(10) == ((10-3)*2)+1
```


In [1]:
def compose(*funcs):
    """
    Return f(x) that applies funcs right-to-left: compose(f,g,h)(x) == f(g(h(x))).
    If no funcs, return identity.
    """
    return funcs[2](funcs[1](funcs[0]))
    ...
id_fn = compose()
assert id_fn(5) == 5
f = compose(lambda x:x+1, lambda x:x*2, lambda x:x-3)
assert f(10) == ((10-3)*2)+1

IndexError: tuple index out of range

In [ ]:
def rect(w, h): return w*h
def rect_kw(**kwargs):
    """Call rect using **kwargs expected to have w and h."""
    return rect(**kwargs)
assert rect_kw(w=3, h=4) == 12

KeyError: 0

In [7]:
# 2
def hypotenuse(a, b):
    return (a*a + b*b) ** 0.5
def hypot_from_pair(pair):
    """Call hypotenuse using *pair."""
    return hypotenuse(*pair)

assert abs(hypot_from_pair((3,4)) - 5) < 1e-9

In [8]:
def sort_by_last_char(words):
    """Return new list sorted by last character; stable; do not mutate input."""
    return sorted(words, key=lambda x: x[-1])
assert sort_by_last_char(["ab","aa","bb"]) == ["aa","ab","bb"]

In [14]:
# 4. 
def apply_pipeline(x, funcs):
    """
    funcs: iterable of callables f: x -> x
    Return result of applying them in order.
    """
    for func in funcs:
        x = func(x)
    return x
assert apply_pipeline(2, [lambda x: x+3, lambda x: x*10]) == 50

In [17]:
from functools import reduce

In [40]:
# 5. 
def transform(nums, pred, mapper):
    """Keep items where pred(x) is True, then map with mapper(x)."""
    return list(map(mapper, filter(lambda x: pred(x), nums)))
assert transform([1,2,3,4], lambda x:x%2==0, lambda x:x*x) == [4,16]


In [41]:
from collections import defaultdict

In [46]:
# 6. 
def group_by(xs, key_fn):
    """Return dict key->list of items in original order."""
    # dct = defaultdict(lambda: defaultdict(str))
    dct = defaultdict(list)
    for x in xs:
        dct[key_fn(x)].append(x)
    return dct
    
g = group_by(["aa","b","ccc","dd"], key_fn=len)
print(g)
assert g == {2:["aa","dd"],1:["b"],3:["ccc"]}

defaultdict(<class 'list'>, {2: ['aa', 'dd'], 1: ['b'], 3: ['ccc']})


In [44]:
from collections import defaultdict
my_list = [1, 3, 2, 5, 2, 4, 5, 1, 2, 4, 3, 5]
counter = defaultdict(int)

for element in my_list:
	counter[str(element)]+=1
print(counter)

defaultdict(<class 'int'>, {'1': 2, '3': 2, '2': 3, '5': 3, '4': 2})


In [52]:
# 7. retry wrapper
def retry(fn, *, attempts=3):
    """
    Call fn(); if it raises, retry up to attempts times.
    Return fn() result or raise last error.
    """
    for i in range(attempts):
        try:
            return fn()
        except Exception as e:
            if i == attempts - 1:  # Last attempt
                raise
count = {"n":0}
def flaky():
    count["n"] += 1
    if count["n"] < 2: raise RuntimeError("boom")
    return "ok"
assert retry(flaky, attempts=3) == "ok"

In [56]:
import time
def timeit(fn):
    """Decorator returning (result, elapsed_seconds)."""
    def wrapper():
        start = time.time()
        result = fn()
        finish = time.time()
        return result, finish - start
    return wrapper
    
@timeit
def slow_add():
    t0 = time.time()
    while time.time() - t0 < 0.01: pass
    return 42
res, secs = slow_add()
assert res == 42 and secs >= 0

In [63]:
def memoize(fn):
    """Decorator caching fn(x) for hashable x."""
    cache = {}
    def wrapper(*args, **kwargs):
        key = (args, tuple(kwargs.items()))
        if key not in cache:
            cache[key] = fn(*args, **kwargs)
        return cache[key]
    return wrapper  # Don't call it!

calls = {"n": 0}

@memoize
def square(x):
    calls["n"] += 1
    return x*x

# assert square(10) == 100 and square(10) == 100 and calls["n"] == 1
print(square(10))
print(square(10))
print(calls["n"])

100
100
1


In [64]:
def compose(*funcs):
    """
    Return f(x) that applies funcs right-to-left: compose(f,g,h)(x) == f(g(h(x))).
    If no funcs, return identity.
    """
    if not funcs:
        return lambda x: x
    
    def composed(x):
        result = x
        for func in reversed(funcs):
            result = func(result)
        return result
    
    return composed

id_fn = compose()
assert id_fn(5) == 5
f = compose(lambda x:x+1, lambda x:x*2, lambda x:x-3)
assert f(10) == ((10-3)*2)+1

In [66]:
def compose(*funcs):
    """
    Return f(x) that applies funcs right-to-left: compose(f,g,h)(x) == f(g(h(x))).
    If no funcs, return identity.
    """
    if not funcs:
        return lambda x: x
    
    return lambda x: reduce(lambda acc, f: f(acc), reversed(funcs), x)

id_fn = compose()
assert id_fn(5) == 5
f = compose(lambda x:x+1, lambda x:x*2, lambda x:x-3)
assert f(10) == ((10-3)*2)+1